# 09 positive friend profile
Reference MAP@200: **0.7278655656**.

Run sections 1–3 to verify saved models. Sections 4–5 rebuild features and retrain. All paths stay inside `models`.

Two fixed seeds: 42 and 2026. All folds use the same fixed parameters.

## 1. Data and shared functions

In [ ]:
from pathlib import Path
import json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'environment/shared.ipynb').exists())
for cell in json.loads((ROOT / 'environment/shared.ipynb').read_text())['cells']:
    if cell['cell_type'] == 'code':
        exec(''.join(cell['source']), globals())
train = pd.read_csv(ROOT / 'data/raw/train.csv', dtype={'user':str,'event':str})
test = pd.read_csv(ROOT / 'data/raw/test.csv', dtype={'user':str,'event':str})
samples = pd.concat([train.drop(columns=['interested','not_interested']),test],ignore_index=True)
folds = pd.read_csv(ROOT / 'data/folds.csv').fold.to_numpy()
HERE = ROOT / '09_positive_friend_profile'
parameters = json.loads((HERE / 'parameters.json').read_text())
seeds = [42, 2026]


## 2. Inspect inputs and fixed configuration

In [ ]:
x = pd.read_pickle(HERE / 'reference/features.pkl')
display(x.head())
display(parameters)


## 3. Reproduce scores from saved models

In [ ]:
scores = evaluate_saved(x, train, folds, HERE / 'reference', seeds, baseline=False)
original = pd.read_csv(HERE / 'reference/oof.csv').probability.to_numpy()
display(pd.Series({'MAP@200':map_users(train,scores).ap200.mean(),'maximum_prediction_difference':np.max(np.abs(scores-original))}))


## 4. Rebuild features from raw tables
This scans the full event table. The reference inputs remain available for exact comparison.

In [ ]:
rebuilt, _ = features(ROOT / 'data/raw', samples)
rebuilt = pd.concat([rebuilt, friend_profile(samples, 'yes')], axis=1)
display(pd.Series({'same_columns':rebuilt.columns.equals(x.columns),'exact_values':rebuilt.equals(x)}))
x = rebuilt


## 5. Retrain five folds
Every fold uses the same saved hyperparameters. Outputs go to `results`; reference artifacts remain unchanged.

In [ ]:
score = train_folds(x, train, test, folds, parameters, seeds, HERE / 'results')
print('MAP@200:', score)


## Verified result

Features rebuilt from raw tables; five folds retrained with the saved parameters.

MAP@200: **0.7278655656**, matching the original experiment.
